# 02 — Grad-CAM: Where Is the Model Looking?

**Goal of this notebook**: for real MRI scans, visualize *which pixels* drove each
model's prediction — not just what class it predicted. This matters for two reasons:

1. **Trust/debugging**: if the model is "right for the wrong reason" (e.g. attending
   to a corner artifact or scanner watermark instead of the tumor), Grad-CAM is how
   you'd catch that — accuracy alone can't tell you.
2. **Comparing the two architectures**: does the custom CNN attend to sensible regions
   as consistently as the fine-tuned DenseNet? This is a qualitative companion to the
   quantitative comparison in `notebooks/06_model_comparison.ipynb`.

**Two different functions are used below, and here's why**: `grad_cam()` works for
flat/Sequential models (the custom CNN). `grad_cam_transfer_model()` is a separate
function specifically for models with a nested pretrained sub-model (DenseNet) — a
Keras-version quirk means an inner layer's output inside a nested sub-model can't be
connected back to the outer model's inputs the normal way. See the docstring in
`src/interpretability.py` for the technical detail; both are tested in
`tests/test_interpretability.py`.

In [ ]:
import sys

sys.path.insert(0, "..")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf

from src.data_utils import load_image_as_array
from src.interpretability import find_last_conv_layer_name, grad_cam, grad_cam_transfer_model

CLASS_NAMES = ["glioma", "meningioma", "pituitary", "no_tumor"]
IMG_SIZE = (224, 224)

In [ ]:
custom_cnn = tf.keras.models.load_model("../models/saved_models/custom_cnn_baseline_best.keras")
densenet_model = tf.keras.models.load_model(
    "../models/saved_models/densenet121_phase2_finetuned_best.keras"
)

## Helper: overlay a heatmap on the original MRI

**Intent**: a raw heatmap array is hard to interpret on its own — overlaying it on
the actual scan (resized to match) makes "where is the model looking" immediately
visible.

In [ ]:
import matplotlib.cm as cm


def overlay_heatmap(image, heatmap, alpha=0.4):
    heatmap_resized = tf.image.resize(heatmap[..., np.newaxis], image.shape[:2]).numpy().squeeze()

    colored_heatmap = cm.jet(heatmap_resized)[:, :, :3]
    base_image = np.repeat(image, 3, axis=-1) if image.shape[-1] == 1 else image

    overlay = (1 - alpha) * base_image + alpha * colored_heatmap
    return np.clip(overlay, 0, 1)

## Pick a few real test-set examples, one per class

**Intent**: look at Grad-CAM across all 4 classes, not just one — a model that
attends sensibly to a glioma but not to a pituitary tumor (a different, smaller
region) would be a real finding, not visible from a single example.

In [ ]:
split_metadata = pd.read_csv("../data/processed/metadata_split.csv")
test_metadata = split_metadata[split_metadata["split"] == "test"]

samples = {
    label: test_metadata[test_metadata["label"] == label].sample(1, random_state=1).iloc[0]
    for label in CLASS_NAMES
    if (test_metadata["label"] == label).any()
}

## Grad-CAM for the custom CNN

In [ ]:
last_conv_custom = find_last_conv_layer_name(custom_cnn)

fig, axes = plt.subplots(1, len(samples), figsize=(4 * len(samples), 4))
for ax, (label, row) in zip(axes, samples.items(), strict=True):
    image = load_image_as_array(row["image_path"], row["source_dataset"], IMG_SIZE)
    heatmap = grad_cam(custom_cnn, image, last_conv_custom)
    overlay = overlay_heatmap(image, heatmap)
    ax.imshow(overlay)
    ax.set_title(f"True: {label}")
    ax.axis("off")
fig.suptitle("Custom CNN — Grad-CAM")
plt.tight_layout()
plt.show()

## Grad-CAM for DenseNet121 (transfer learning)

**Intent**: same samples, same visualization — direct visual comparison against the
custom CNN above. Note the 3-channel repeat, matching how this model was trained
(see `notebooks/05_transfer_learning_densenet.ipynb`).

In [ ]:
fig, axes = plt.subplots(1, len(samples), figsize=(4 * len(samples), 4))
for ax, (label, row) in zip(axes, samples.items(), strict=True):
    image = load_image_as_array(row["image_path"], row["source_dataset"], IMG_SIZE)
    image_3ch = np.repeat(image, 3, axis=-1)
    heatmap = grad_cam_transfer_model(densenet_model, image_3ch)
    overlay = overlay_heatmap(image, heatmap)
    ax.imshow(overlay)
    ax.set_title(f"True: {label}")
    ax.axis("off")
fig.suptitle("DenseNet121 (transfer) — Grad-CAM")
plt.tight_layout()
plt.show()

## Conclusion

Fill in after running: for each class, does the highlighted (red/yellow) region
actually overlap with where a tumor would be expected, or does it drift to image
borders / unrelated structures? Note any class where one model looks noticeably more
"localized" than the other — that's a concrete, visual finding worth including in
your report alongside the accuracy numbers.

Next notebook: **03_training_dynamics_comparison.ipynb** — look at *how* each model's
learning unfolded over training, including the visible effect of the phase 1 → phase 2
unfreezing step for DenseNet.